# Resolve one product

Mandatory inputs: `main_text`, `country_code`. Optional inputs: `ean`, `retailer_name`.

This notebook uses native top-level `await`. It does not start a UI/API service and does not patch the Jupyter event loop.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file()
)
os.chdir(ROOT)
load_dotenv(ROOT / ".env", override=False)

from product_url import Budgets, ProductInput, ProductURLResolver

print(f"Repository: {ROOT}")
print("SERPAPI_API_KEY configured:", bool(os.getenv("SERPAPI_API_KEY")))
print("PCA reasoning enabled:", os.getenv("PRODUCT_URL_REASONING_ENABLED", "true"))

## Budgets

These are the hard maximums for one product.

In [ ]:
BUDGETS = Budgets(
    searches=3,
    search_results=10,
    crawls=6,
    llm_calls=2,
)
BUDGETS

## Product input

In [ ]:
PRODUCT = ProductInput(
    row_id="DEMO-1",
    main_text="PKM ME04 WACHSENDES CHAOS BOOSTER",
    country_code="CH",
    ean="196214141070",
    retailer_name=None,
)
PRODUCT

## Run

In [ ]:
async with ProductURLResolver(
    budgets=BUDGETS,
    artifact_root=ROOT / "data" / "artifacts",
) as resolver:
    RESULT = await resolver.resolve(PRODUCT)

pd.DataFrame([RESULT["output"]])

## Candidate evidence

In [ ]:
candidate_columns = [
    "id", "url", "final_url", "title", "page_title", "source", "rank",
    "crawl_success", "identity_score", "product_page",
    "identifier_match", "retailer_match", "identity_status",
    "blockers", "crawl_error",
]
pd.DataFrame(RESULT["candidates"]).reindex(columns=candidate_columns)

## Search and LLM usage

In [ ]:
print("Queries:")
for query in RESULT["queries"]:
    print("-", query)
print("\nUsage:", RESULT["usage"])
print("\nHypothesis:")
print(RESULT["hypothesis"])
print("\nLLM choice:")
print(RESULT["llm_choice"])

## Audit artifact

In [ ]:
audit_path = Path(RESULT["artifact_dir"]) / "audit.md"
print(audit_path.read_text(encoding="utf-8"))